In [ ]:
"""
SBCK bias correction with sliding windows (30 years)

Author: Hajar BADRAOUI

Date : 30/03/2026
"""

import sys
sys.path.append('.')

import numpy as np
from SBCK import CDFt
from iriscc.settings import DATASET_BC_DIR

np.random.seed(42)

# =========================
# 1. LOAD DATA (NPZ FORMAT)
# =========================
# data is already structured as 3D arrays (time, lat, lon)

train_hist = dict(np.load(DATASET_BC_DIR/'bc_train_hist.npz', allow_pickle=True))
test_future = dict(np.load(DATASET_BC_DIR/'bc_test_future.npz', allow_pickle=True))

# Observations (ERA5 / SAFRAN equivalent)
Y0 = train_hist['era5']   # shape: (time, lat, lon)

# Historical model
X0 = train_hist['gcm']

# Future model
X1 = test_future['gcm']

time_len, lat, lon = Y0.shape


# =========================
# 2. SSR FUNCTIONS (ONLY FOR PRECIPITATION)
# =========================
def log1x(x):
    """
    Log-like transformation adapted for precipitation.
    Avoids log(0) issues.
    """
    return np.where(x >= 1, x - 1,
           np.where((x > 0) & (x < 1), np.log(x), 0))


def exp1x(x):
    """
    Inverse transformation of log1x.
    """
    return np.where(x < 0, np.exp(x), x + 1)


def apply_ssr(data, th):
    """
Applies SSR (Singularity stochastic Removal) to a precipitation time series.

Purpose:
    Replace zero values with very small strictly positive random values
    drawn from a uniform distribution in the interval (1e-6, th).
    This avoids issues related to logarithmic transformation,
    since log(0) is undefined.

Parameters:
    data (array-like): numpy array containing precipitation data.
    th (float): fixed positive threshold, typically defined as the minimum
                strictly positive value across both model data and reference observations.

Returns:
    np.ndarray: corrected array where all zero values have been replaced
                by small random positive values.
    """
    data = data.copy()
    idx = np.where(data == 0)[0]
    if len(idx) > 0:
        data[idx] = np.random.uniform(1e-6, th, size=len(idx))
    return data


# =========================
# 3. SLIDING WINDOWS (30 YEARS)
# =========================
"""
Each tuple:
(full_start, full_end, corr_start, corr_end)

- full window (30 years) used for training CDF-t
- central window (20 years) where correction is applied
"""
windows = [
    (0, 30, 0, 20),
    (10, 40, 20, 30),
    (20, 50, 30, 40),
]


# =========================
# 4. OUTPUT ARRAY
# =========================
# Same shape as future model
Z_final = np.zeros_like(X1)


# =========================
# 5. MAIN LOOP (GRID-BASED)
# =========================
print("Starting SBCK correction...")

# Loop over each grid cell (lat, lon)
for i in range(lat):
    for j in range(lon):

        # Extract time series at one grid cell
        y0 = Y0[:, i, j]
        x0 = X0[:, i, j]
        x1 = X1[:, i, j]

        # Skip invalid cells
        if np.all(np.isnan(y0)) or np.all(np.isnan(x0)):
            continue

        # =========================
        # 5.1 SSR (ONLY FOR PRECIPITATION)
        # =========================
        is_precip = True  #  change to False for tas, hurs, etc.

        if is_precip:
            all_vals = np.concatenate([
                y0[y0 > 0],
                x0[x0 > 0],
                x1[x1 > 0]
            ])

            if len(all_vals) == 0:
                continue

            th = np.min(all_vals)

            y0 = apply_ssr(y0, th)
            x0 = apply_ssr(x0, th)
            x1 = apply_ssr(x1, th)

        # =========================
        # 5.2 SLIDING WINDOWS
        # =========================
        for full_start, full_end, corr_start, corr_end in windows:

            if full_end > len(x1):
                continue

            # Full window (training)
            x1_full = x1[full_start:full_end]

            # Central window (correction)
            x1_corr = x1[corr_start:corr_end]

            if len(x1_corr) == 0:
                continue

            # =========================
            # 5.3 MONTHLY CORRECTION
            # =========================
            for m in range(12):

                # Select indices for month m
                idx_y0 = np.arange(len(y0))[np.arange(len(y0)) % 12 == m]
                idx_x0 = np.arange(len(x0))[np.arange(len(x0)) % 12 == m]
                idx_x1 = np.arange(len(x1_corr))[np.arange(len(x1_corr)) % 12 == m]

                if len(idx_x1) == 0:
                    continue

                y0_m = y0[idx_y0]
                x0_m = x0[idx_x0]
                x1_m = x1_corr[idx_x1]

                # =========================
                # 5.4 RESHAPE FOR SBCK
                # =========================
                if is_precip:
                    y0_m = log1x(y0_m).reshape(-1, 1)
                    x0_m = log1x(x0_m).reshape(-1, 1)
                    x1_m = log1x(x1_m).reshape(-1, 1)
                else:
                    y0_m = y0_m.reshape(-1, 1)
                    x0_m = x0_m.reshape(-1, 1)
                    x1_m = x1_m.reshape(-1, 1)

                # =========================
                # 5.5 SBCK CDF-t
                # =========================
                cdft = CDFt(version=2, normalize_cdf=True)

                # Fit using obs + hist + future
                cdft.fit(y0_m, x0_m, x1_m)

                # Predict corrected future
                Z1, _ = cdft.predict(x1_m, x0_m)
                Z1 = Z1[:, 0]

                # =========================
                # 5.6 INVERSE SSR
                # =========================
                if is_precip:
                    Z1 = exp1x(Z1)
                    Z1[Z1 < th] = 0

                # =========================
                # 5.7 SIZE SAFETY
                # =========================
                Z1 = Z1[:len(idx_x1)]

                # =========================
                # 5.8 ASSIGN VALUES
                # =========================
                Z_final[corr_start:corr_end, i, j][idx_x1] = Z1


print("SBCK DONE")


# =========================
# 6. SAVE OUTPUT (NPZ)
# =========================
np.savez(DATASET_BC_DIR/'bc_future_sbck_sliding.npz', data=Z_final)

print("Saved ")